<a href="https://colab.research.google.com/github/Heena667/Demo-/blob/main/creditcardfraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy scikit-learn joblib streamlit matplotlib

In [ ]:
%%writefile generate_data.py
"""Create a synthetic credit-card transaction dataset (~1% fraud)."""
import numpy as np, pandas as pd, os

rng = np.random.default_rng(42)
N = 100_000
fraud = rng.random(N) < 0.01

cust_avg = rng.gamma(5, 12, N)
amount = np.where(fraud, cust_avg * rng.uniform(3, 10, N), rng.gamma(2, cust_avg / 2))
hour = np.where(fraud, rng.choice([0, 1, 2, 3, 4, 23], N), rng.integers(7, 23, N))

df = pd.DataFrame({
    "TRANSACTION_ID": np.arange(N),
    "TX_DATETIME": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 180 * 86400, N), unit="s"),
    "CUSTOMER_ID": rng.integers(1, 5000, N),
    "TERMINAL_ID": rng.integers(1, 1000, N),
    "TX_AMOUNT": amount.round(2),
    "HOUR": hour,
    "DAY_OF_WEEK": rng.integers(0, 7, N),
    "CUSTOMER_AVG_AMOUNT": cust_avg.round(2),
    "DISTANCE_FROM_HOME": np.where(fraud, rng.uniform(100, 2000, N), rng.exponential(10, N)).round(2),
    "DEVICE_RISK": np.where(fraud, rng.beta(5, 2, N), rng.beta(2, 5, N)).round(3),
    "MERCHANT_RISK": np.where(fraud, rng.beta(5, 2, N), rng.beta(2, 5, N)).round(3),
    "FAILED_ATTEMPTS": np.where(fraud, rng.integers(1, 6, N), rng.choice([0, 0, 0, 1], N)),
    "TX_FRAUD": fraud.astype(int),
})
df["AMOUNT_DEVIATION"] = (df.TX_AMOUNT - df.CUSTOMER_AVG_AMOUNT).round(2)

os.makedirs("data", exist_ok=True)
df.to_csv("data/transactions.csv", index=False)
print(df.shape, "fraud rate:", df.TX_FRAUD.mean().round(4))

In [ ]:
%%writefile train.py
"""Train Isolation Forest, evaluate, save model."""
import pandas as pd, joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = ["TX_AMOUNT", "HOUR", "DAY_OF_WEEK", "CUSTOMER_AVG_AMOUNT",
            "AMOUNT_DEVIATION", "DISTANCE_FROM_HOME", "DEVICE_RISK",
            "MERCHANT_RISK", "FAILED_ATTEMPTS"]

df = pd.read_csv("data/transactions.csv")
print("Shape:", df.shape, "| Missing:", df.isna().sum().sum(), "| Duplicates:", df.duplicated().sum())
print(df.TX_FRAUD.value_counts(normalize=True))

X, y = df[FEATURES], df["TX_FRAUD"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler().fit(X_tr)
model = IsolationForest(n_estimators=200, contamination=0.01, random_state=42, n_jobs=-1)
model.fit(scaler.transform(X_tr))

pred = (model.predict(scaler.transform(X_te)) == -1).astype(int)   # -1 = anomaly = fraud
print(confusion_matrix(y_te, pred))
print(classification_report(y_te, pred, target_names=["Normal", "Fraud"]))

joblib.dump({"model": model, "scaler": scaler, "features": FEATURES}, "model.joblib")
print("Saved model.joblib")

In [ ]:
!python generate_data.py
!python train.py

In [ ]:
%%writefile app.py
import streamlit as st, pandas as pd, joblib

st.set_page_config(page_title="Fraud Detection", page_icon="💳")
st.title("💳 Credit Card Fraud Detection")
st.caption("Isolation Forest anomaly detection")

@st.cache_resource
def load(): return joblib.load("model.joblib")
b = load()

def score(df):
    X = b["scaler"].transform(df[b["features"]])
    out = df.copy()
    out["ANOMALY_SCORE"] = -b["model"].score_samples(X)
    out["PREDICTION"] = ["Fraud" if p == -1 else "Normal" for p in b["model"].predict(X)]
    return out

tab1, tab2 = st.tabs(["Single transaction", "Batch CSV"])

with tab1:
    c1, c2 = st.columns(2)
    amt = c1.number_input("Amount", 0.0, 100000.0, 50.0)
    avg = c2.number_input("Customer avg amount", 0.0, 100000.0, 60.0)
    hour = c1.slider("Hour", 0, 23, 14)
    dow = c2.slider("Day of week (0=Mon)", 0, 6, 2)
    dist = c1.number_input("Distance from home (km)", 0.0, 20000.0, 5.0)
    fails = c2.number_input("Failed attempts", 0, 20, 0)
    dev = c1.slider("Device risk", 0.0, 1.0, 0.2)
    mer = c2.slider("Merchant risk", 0.0, 1.0, 0.2)
    if st.button("Check transaction"):
        r = score(pd.DataFrame([{
            "TX_AMOUNT": amt, "HOUR": hour, "DAY_OF_WEEK": dow, "CUSTOMER_AVG_AMOUNT": avg,
            "AMOUNT_DEVIATION": amt - avg, "DISTANCE_FROM_HOME": dist, "DEVICE_RISK": dev,
            "MERCHANT_RISK": mer, "FAILED_ATTEMPTS": fails}])).iloc[0]
        (st.error if r.PREDICTION == "Fraud" else st.success)(
            f"{r.PREDICTION} — anomaly score {r.ANOMALY_SCORE:.3f}")

with tab2:
    f = st.file_uploader("Upload CSV with the 9 feature columns", type="csv")
    if f:
        res = score(pd.read_csv(f))
        st.write(res.PREDICTION.value_counts())
        st.dataframe(res.sort_values("ANOMALY_SCORE", ascending=False).head(100))
        st.download_button("Download results", res.to_csv(index=False), "results.csv")

In [ ]:
import joblib, pandas as pd
b = joblib.load("model.joblib")
tx = {"TX_AMOUNT": 900, "HOUR": 3, "DAY_OF_WEEK": 2, "CUSTOMER_AVG_AMOUNT": 60,
      "AMOUNT_DEVIATION": 840, "DISTANCE_FROM_HOME": 800, "DEVICE_RISK": 0.9,
      "MERCHANT_RISK": 0.8, "FAILED_ATTEMPTS": 3}
X = b["scaler"].transform(pd.DataFrame([tx]))
print("FRAUD" if b["model"].predict(X)[0] == -1 else "NORMAL", -b["model"].score_samples(X)[0])

In [ ]:
!npm install -q localtunnel
import urllib
print("Password / IP for the tunnel page:", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!streamlit run app.py &>/content/logs.txt &
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.2
npm notice To update run: npm install -g npm@12.2.2
npm notice
⠋Password / IP for the tunnel page: 136.67.175.130
⠙your url is: https://metal-pens-teach.loca.lt
